In [77]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [78]:
df = pd.read_csv('data/ps2_ex2.csv')

In [79]:
K = 10
df['diff'] = df.milage.diff().shift(-1)
df['action'] = (df['diff'] < 0).astype(int)
# Get rid of diff column
df.drop(columns=['diff'], inplace=True)

In [81]:
# Discretize milage into bins
bins = np.linspace(0, replace_df["milage"].max(), K + 1)
df["milage_bin"] = pd.cut(df["milage"], bins=bins, labels=False, include_lowest=True)

# Subtract 1 from milage_bin to make it 0-indexed
df["milage_bin"] = df["milage_bin"].astype(int)

# Create column that indicates next state
df['next_milage_bin'] = df.milage_bin.shift(-1)

# Drop last row because of nan
df.dropna(inplace=True)

# Convert all bins to ints
df['next_milage_bin'] = df['next_milage_bin'].astype(int)

In [82]:
replace_df = df[df.action == 1].copy()
cont_df = df[df.action == 0].copy()

In [86]:
# Construct transition matrix
replace_trans_mx = np.zeros((K, K))
cont_trans_mx = np.zeros((K, K)) # the first element is the transition probabilties from state 0 to all states.
milage_bins = np.arange(K)
for i in range(K):
    # Continue transition matrix
    temp_df = cont_df[cont_df.milage_bin == i]
    temp_value_counts = temp_df.next_milage_bin.value_counts().reset_index()
    
    # Ensure all states are represented with 10 rows
    all_states = pd.DataFrame({'next_milage_bin': milage_bins})
    temp_value_counts = all_states.merge(temp_value_counts, on='next_milage_bin', how='left').fillna(0)
    temp_value_counts['count'] = temp_value_counts['count'].astype(int)
    
    tot_sum = temp_value_counts['count'].sum()
    temp_value_counts['prob'] = temp_value_counts['count'] / tot_sum if tot_sum > 0 else 0
    cont_trans_mx[i] = temp_value_counts['prob'].values
    
    # Replace transition matrix
    temp_df = replace_df[replace_df.milage_bin == i]
    temp_value_counts = temp_df.next_milage_bin.value_counts().reset_index()
    
    # Ensure all states are represented with 10 rows
    all_states = pd.DataFrame({'next_milage_bin': milage_bins})
    temp_value_counts = all_states.merge(temp_value_counts, on='next_milage_bin', how='left').fillna(0)
    temp_value_counts['count'] = temp_value_counts['count'].astype(int)
    
    tot_sum = temp_value_counts['count'].sum()
    temp_value_counts['prob'] = temp_value_counts['count'] / tot_sum if tot_sum > 0 else 0
    replace_trans_mx[i] = temp_value_counts['prob'].values

In [87]:
cont_trans_df = pd.DataFrame(cont_trans_mx)
replace_trans_df = pd.DataFrame(replace_trans_mx)

In [88]:
replace_trans_df


,0,1,2,3,4,5,6,7,8,9
0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.977273,0.022727,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.989583,0.010417,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.990196,0.009804,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,0.988764,0.011236,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,1.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,1.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,1.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
